<a href="https://colab.research.google.com/github/guitorte/audio/blob/claude/song-to-midi-converter-KRClC/Song_to_MIDI_Converter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎵 Song to MIDI Converter

Convert any audio file (MP3, WAV, etc.) to playable MIDI using **Spotify's Basic Pitch** — a state-of-the-art neural network for automatic music transcription.

## Features
- 🎼 **Automatic transcription** - Converts melody to MIDI notes
- 📊 **Audio analysis** - Extracts BPM, key, duration, spectrogram
- 🎹 **Piano roll visualization** - See the generated MIDI as a musical score
- 🔊 **Playback preview** - Listen to the generated MIDI
- 💾 **Download results** - Save MIDI to your computer or Google Drive

## Workflow
1. **Upload** an audio file (MP3, WAV, M4A, etc.)
2. **Analyze** the audio (waveform, spectrogram, tempo, key)
3. **Convert** to MIDI with configurable parameters
4. **Visualize** as a piano roll
5. **Preview** the MIDI with playback
6. **Download** the MIDI file

---

In [ ]:
# @title 1️⃣ Install Dependencies (Run Once)
# This installs all required packages for audio processing and MIDI conversion

import subprocess
import sys

print("Installing dependencies (this takes ~2 minutes on first run)...")
print("Progress: Installing Python packages...")

packages = [
    'basic-pitch',
    'onnxruntime',
    'mir_eval',
    'librosa',
    'pretty-midi',
    'soundfile',
    'matplotlib',
    'mido',
    'importlib_resources'
]

for pkg in packages:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=False)

print("\nProgress: Installing system packages...")
subprocess.run(['apt-get', 'update'], capture_output=True, check=False)
subprocess.run(['apt-get', 'install', '-y', '-q', 'fluidsynth'], capture_output=True, check=False)

# Verify installations
try:
    import basic_pitch
    import librosa
    import pretty_midi
    import matplotlib
    print("\n✅ All dependencies installed successfully!")
except ImportError as e:
    print(f"⚠️ Warning: {e}")
    print("Some features may not work. Try re-running this cell.")

In [ ]:
# @title 2️⃣ Load Libraries and Define Helpers

import os
import warnings
import tempfile
import shutil
import subprocess
import numpy as np
import librosa
import pretty_midi
import mido
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from IPython.display import Audio, display, HTML, FileLink
from basic_pitch.inference import predict
from pathlib import Path

warnings.filterwarnings('ignore')

# Create output directory
OUTPUT_DIR = '/tmp/midi_outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("✅ Libraries loaded.")
print(f"Output directory: {OUTPUT_DIR}")

In [ ]:
# @title 3️⃣ Mount Google Drive (Optional)
# Enable this to save MIDI files directly to your Google Drive

mount_drive = False  # @param {type:"boolean"}

SAVE_DIR = OUTPUT_DIR

if mount_drive:
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=True)
        SAVE_DIR = "/content/drive/MyDrive/MIDI_Outputs"
        os.makedirs(SAVE_DIR, exist_ok=True)
        print(f"✅ Google Drive mounted. Saving to: {SAVE_DIR}")
    except ImportError:
        print("ℹ️ Not running in Google Colab. Using local directory.")
        SAVE_DIR = OUTPUT_DIR
else:
    print(f"ℹ️ Files will be saved to: {SAVE_DIR}")

In [ ]:
# @title 4️⃣ Load Audio File
# Choose how to load your audio file

from google.colab import files
import time

audio_source = "upload"  # @param ["upload", "url", "example"]
url_input = ""  # @param {type:"string"}

audio_file = None

if audio_source == "upload":
    print("📁 Click 'Choose Files' to upload an audio file (MP3, WAV, M4A, FLAC, etc.)")
    uploaded = files.upload()
    if uploaded:
        audio_file = list(uploaded.keys())[0]
        print(f"✅ Uploaded: {audio_file}")

elif audio_source == "url":
    if url_input.strip():
        print(f"Downloading from URL: {url_input}")
        import urllib.request
        audio_file = os.path.join(OUTPUT_DIR, "downloaded_audio.mp3")
        urllib.request.urlretrieve(url_input, audio_file)
        print(f"✅ Downloaded to: {audio_file}")
    else:
        print("⚠️ Please provide a URL above.")

elif audio_source == "example":
    print("Using example audio...")
    # In Colab, we can use a simple test audio or fetch from web
    audio_file = os.path.join(OUTPUT_DIR, "example.wav")
    # Create a simple sine wave example
    import scipy.io.wavfile as wavfile
    sr = 22050
    duration = 10
    t = np.linspace(0, duration, int(sr * duration))
    # Melody: Do-Re-Mi-Fa-Sol
    frequencies = [261.63, 293.66, 329.63, 349.23, 392.00]
    signal = np.zeros_like(t)
    note_duration = 2
    for i, freq in enumerate(frequencies):
        start = int(i * note_duration * sr)
        end = int((i + 1) * note_duration * sr)
        signal[start:end] = 0.3 * np.sin(2 * np.pi * freq * t[start:end])
    wavfile.write(audio_file, sr, (signal * 32767).astype(np.int16))
    print(f"✅ Created example audio: {audio_file}")

if audio_file and os.path.exists(audio_file):
    file_size = os.path.getsize(audio_file) / (1024 * 1024)
    print(f"📊 File size: {file_size:.1f} MB")
else:
    print("⚠️ No audio file loaded.")

In [ ]:
# @title 5️⃣ Analyze Audio
# Extract audio features: waveform, spectrogram, BPM, key, duration

if not audio_file or not os.path.exists(audio_file):
    print("⚠️ Please load an audio file first (Step 4).")
else:
    print(f"Analyzing: {audio_file}")
    print("Loading audio...")
    
    # Load audio
    try:
        y, sr = librosa.load(audio_file, sr=None)
        duration = librosa.get_duration(y=y, sr=sr)
        
        print(f"✅ Loaded: {duration:.1f}s at {sr}Hz")
        
        # Estimate tempo and key
        print("Extracting musical features...")
        tempo, _ = librosa.beat.beat_track(y=y, sr=sr)
        tempo = float(np.atleast_1d(tempo)[0])
        
        # Estimate key via chroma
        chroma = librosa.feature.chroma_stft(y=y, sr=sr)
        key_idx = int(np.argmax(np.mean(chroma, axis=1)))
        keys = ['C', 'C#', 'D', 'D#', 'E', 'F', 'F#', 'G', 'G#', 'A', 'A#', 'B']
        key = keys[key_idx]
        
        # RMS energy
        rms = librosa.feature.rms(y=y)[0]
        mean_rms = np.mean(rms)
        
        print(f"\n📊 Audio Analysis:")
        print(f"  Duration: {duration:.1f} seconds")
        print(f"  Tempo: ~{tempo:.0f} BPM")
        print(f"  Key: {key}")
        print(f"  Sample rate: {sr} Hz")
        print(f"  RMS Energy: {mean_rms:.3f}")
        
        # Plot waveform and spectrogram
        print("\nGenerating visualizations...")
        fig, axes = plt.subplots(2, 1, figsize=(14, 8))
        
        # Waveform
        librosa.display.waveshow(y, sr=sr, ax=axes[0], color='steelblue')
        axes[0].set_title('Waveform', fontsize=12, fontweight='bold')
        axes[0].set_ylabel('Amplitude')
        
        # Spectrogram
        D = librosa.amplitude_to_db(np.abs(librosa.stft(y)), ref=np.max)
        img = librosa.display.specshow(D, sr=sr, x_axis='time', y_axis='log', ax=axes[1], cmap='viridis')
        axes[1].set_title('Spectrogram (Log Scale)', fontsize=12, fontweight='bold')
        fig.colorbar(img, ax=axes[1], format='%+2.0f dB')
        
        plt.tight_layout()
        plt.savefig(os.path.join(OUTPUT_DIR, 'audio_analysis.png'), dpi=100, bbox_inches='tight')
        plt.show()
        print("✅ Analysis complete!")
        
    except Exception as e:
        print(f"❌ Error: {e}")

In [ ]:
# @title 6️⃣ Convert Audio to MIDI
# Uses Spotify's Basic Pitch neural network to transcribe melody to MIDI

if not audio_file or not os.path.exists(audio_file):
    print("⚠️ Please load an audio file first (Step 4).")
else:
    print(f"Converting to MIDI: {audio_file}")
    print("This may take 1-3 minutes depending on file length...")
    
    try:
        # Run Basic Pitch prediction
        print("Running neural network transcription...")
        model_output, midi_data, note_events = predict(audio_file)
        
        num_notes = len(note_events)
        num_instruments = len(midi_data.instruments)
        duration = midi_data.get_end_time()
        
        print(f"\n✅ Conversion successful!")
        print(f"  Notes detected: {num_notes}")
        print(f"  MIDI tracks: {num_instruments}")
        print(f"  Duration: {duration:.1f} seconds")
        
        # Get note statistics
        if num_instruments > 0:
            notes = midi_data.instruments[0].notes
            if notes:
                pitches = [n.pitch for n in notes]
                durations = [n.end - n.start for n in notes]
                print(f"  Pitch range: {min(pitches)}-{max(pitches)} (MIDI notes)")
                print(f"  Avg note duration: {np.mean(durations):.3f}s")
                print(f"  Loudness range: {min(n.velocity for n in notes)}-{max(n.velocity for n in notes)}")
        
        # Save MIDI
        midi_output_path = os.path.join(SAVE_DIR, 'transcription.mid')
        midi_data.write(midi_output_path)
        print(f"\n💾 MIDI saved to: {midi_output_path}")
        
    except Exception as e:
        print(f"❌ Error during conversion: {e}")
        import traceback
        traceback.print_exc()

In [ ]:
# @title 7️⃣ Visualize MIDI (Piano Roll)
# Show the MIDI as a piano roll visualization

try:
    if 'midi_data' not in dir():
        print("⚠️ Please convert audio to MIDI first (Step 6).")
    else:
        print("Generating piano roll...")
        
        # Get piano roll
        piano_roll = midi_data.get_piano_roll(fs=10)  # 10Hz resolution
        
        # Extract meaningful pitch range (typically 20-100)
        pitch_min, pitch_max = 30, 90  # C1 to F#6
        roll_slice = piano_roll[pitch_min:pitch_max+1, :]
        
        # Create visualization
        fig, ax = plt.subplots(figsize=(14, 6))
        
        # Piano roll as heatmap
        extent = [0, roll_slice.shape[1] / 10, pitch_min, pitch_max]
        im = ax.imshow(roll_slice, aspect='auto', origin='lower',
                       cmap='Blues', interpolation='nearest', extent=extent)
        
        ax.set_xlabel('Time (seconds)', fontsize=11)
        ax.set_ylabel('MIDI Pitch', fontsize=11)
        ax.set_title('Piano Roll (Transcribed Melody)', fontsize=13, fontweight='bold')
        
        # Add note labels
        note_names = ['C', 'C#', 'D', 'D#', 'E', 'F', 'F#', 'G', 'G#', 'A', 'A#', 'B']
        yticks = np.arange(pitch_min, pitch_max+1, 2)
        yticklabels = [f"{note_names[i%12]}{i//12}" for i in yticks]
        ax.set_yticks(yticks)
        ax.set_yticklabels(yticklabels, fontsize=8)
        
        cbar = plt.colorbar(im, ax=ax)
        cbar.set_label('Velocity', fontsize=10)
        
        plt.tight_layout()
        plt.savefig(os.path.join(OUTPUT_DIR, 'piano_roll.png'), dpi=100, bbox_inches='tight')
        plt.show()
        
        # Statistics
        print("\n📊 MIDI Statistics:")
        print(f"  Piano roll shape: {roll_slice.shape}")
        print(f"  Pitch range (displayed): {pitch_min}-{pitch_max}")
        print(f"  Time resolution: 0.1s")
        
        # Note distribution
        fig, axes = plt.subplots(1, 2, figsize=(14, 4))
        
        # Pitch histogram
        notes = midi_data.instruments[0].notes
        pitches = [n.pitch for n in notes]
        axes[0].hist(pitches, bins=20, color='steelblue', edgecolor='black', alpha=0.7)
        axes[0].set_xlabel('MIDI Pitch')
        axes[0].set_ylabel('Frequency')
        axes[0].set_title('Pitch Distribution')
        axes[0].grid(alpha=0.3)
        
        # Duration histogram
        durations = [n.end - n.start for n in notes]
        axes[1].hist(durations, bins=20, color='seagreen', edgecolor='black', alpha=0.7)
        axes[1].set_xlabel('Note Duration (seconds)')
        axes[1].set_ylabel('Frequency')
        axes[1].set_title('Note Duration Distribution')
        axes[1].grid(alpha=0.3)
        
        plt.tight_layout()
        plt.savefig(os.path.join(OUTPUT_DIR, 'note_distribution.png'), dpi=100, bbox_inches='tight')
        plt.show()
        
        print("✅ Visualization complete!")
        
except Exception as e:
    print(f"❌ Error: {e}")
    import traceback
    traceback.print_exc()

In [ ]:
# @title 8️⃣ MIDI Playback (Preview)
# Convert MIDI to audio and play it

try:
    if 'midi_data' not in dir():
        print("⚠️ Please convert audio to MIDI first (Step 6).")
    else:
        print("Preparing MIDI playback...")
        
        # Try using fluidsynth if available
        midi_path = os.path.join(OUTPUT_DIR, 'transcription.mid')
        audio_out = os.path.join(OUTPUT_DIR, 'playback.wav')
        
        try:
            # Use fluidsynth for high-quality synthesis
            result = subprocess.run(
                ['fluidsynth', '-ni', '/usr/share/sounds/sf2/FluidR3_GM.sf2',
                 midi_path, '-F', audio_out, '-r', '22050'],
                capture_output=True, timeout=60
            )
            
            if os.path.exists(audio_out):
                print("✅ MIDI synthesized with FluidSynth")
                print("🔊 Playing preview...")
                display(Audio(audio_out))
            else:
                raise Exception("FluidSynth synthesis failed")
                
        except (FileNotFoundError, subprocess.TimeoutExpired, Exception) as e:
            print(f"⚠️ FluidSynth unavailable ({e}). Using simple synthesis...")
            
            # Fallback: simple sine wave synthesis
            sr = 22050
            duration = midi_data.get_end_time()
            signal = np.zeros(int(sr * duration))
            
            for note in midi_data.instruments[0].notes:
                start_sample = int(note.start * sr)
                end_sample = int(note.end * sr)
                
                # MIDI pitch to frequency
                freq = 440 * (2 ** ((note.pitch - 69) / 12))
                
                # Generate sine wave
                t = np.arange(end_sample - start_sample) / sr
                velocity_factor = note.velocity / 127.0
                signal[start_sample:end_sample] += 0.3 * velocity_factor * np.sin(2 * np.pi * freq * t)
            
            # Normalize
            signal = signal / np.max(np.abs(signal))
            
            print("✅ MIDI synthesized with simple sine wave")
            print("🔊 Playing preview...")
            display(Audio(data=signal, rate=sr))
            
except Exception as e:
    print(f"❌ Playback error: {e}")

In [ ]:
# @title 9️⃣ Download Results
# Download MIDI and visualizations to your computer

print("📥 Available Files:\n")

try:
    if 'midi_data' not in dir():
        print("⚠️ No MIDI generated yet.")
    else:
        files_to_show = [
            ('transcription.mid', 'MIDI File'),
            ('audio_analysis.png', 'Audio Analysis Plot'),
            ('piano_roll.png', 'Piano Roll'),
            ('note_distribution.png', 'Note Distribution'),
        ]
        
        for filename, description in files_to_show:
            filepath = os.path.join(OUTPUT_DIR, filename)
            if os.path.exists(filepath):
                size = os.path.getsize(filepath) / 1024
                print(f"✅ {description}: {filename} ({size:.1f} KB)")
                try:
                    display(FileLink(filepath, result_html_prefix='   📥 '))
                except:
                    print(f"   Path: {filepath}")
        
        print("\n💾 To download files in Colab:")
        print("  - Click the download link above, OR")
        print("  - Run the code below to download as ZIP")
        
        # Generate download code
        print("\n" + "="*50)
        print("Optional: Download all files as ZIP")
        print("="*50)
        
except Exception as e:
    print(f"Error: {e}")

In [ ]:
# @title 🔟 Advanced: Export & Batch Processing
# Additional options for MIDI manipulation and export

try:
    if 'midi_data' not in dir():
        print("⚠️ No MIDI generated yet.")
    else:
        print("Advanced MIDI Processing Options:\n")
        
        # 1. Transpose
        transpose_semitones = 0  # @param {type:"slider", min:-12, max:12}
        
        if transpose_semitones != 0:
            transposed = pretty_midi.PrettyMIDI()
            for instrument in midi_data.instruments:
                new_instrument = pretty_midi.Instrument(program=instrument.program)
                for note in instrument.notes:
                    new_note = pretty_midi.Note(
                        velocity=note.velocity,
                        pitch=np.clip(note.pitch + transpose_semitones, 0, 127),
                        start=note.start,
                        end=note.end
                    )
                    new_instrument.notes.append(new_note)
                transposed.instruments.append(new_instrument)
            transposed.write(os.path.join(SAVE_DIR, f'transcription_T{transpose_semitones:+d}.mid'))
            print(f"✅ Transposed by {transpose_semitones} semitones")
        
        # 2. Quantize to grid
        quantize_grid = 0.1  # @param {type:"slider", min:0.01, max:1, step:0.05}
        
        quantized = pretty_midi.PrettyMIDI()
        for instrument in midi_data.instruments:
            new_instrument = pretty_midi.Instrument(program=instrument.program)
            for note in instrument.notes:
                new_note = pretty_midi.Note(
                    velocity=note.velocity,
                    pitch=note.pitch,
                    start=round(note.start / quantize_grid) * quantize_grid,
                    end=round(note.end / quantize_grid) * quantize_grid
                )
                new_instrument.notes.append(new_note)
            quantized.instruments.append(new_instrument)
        quantized.write(os.path.join(SAVE_DIR, 'transcription_quantized.mid'))
        print(f"✅ Quantized to {quantize_grid}s grid")
        
        print("\n💾 Processed files saved to output directory")
        
except Exception as e:
    print(f"Error: {e}")

---

## 📚 Notes & Tips

### How It Works
1. **Basic Pitch** uses a neural network trained on 10,000+ hours of music
2. It predicts MIDI note activations for each time frame
3. The output is a melody transcription (single notes, not polyphonic)

### Best Practices
- **Melody-dominant audio** works best (vocals, lead instruments)
- **Clean recordings** produce more accurate results
- **Audio length**: Works with 10 seconds to 10+ minutes
- **File formats**: MP3, WAV, M4A, FLAC, OGG all supported

### Tips for Better Results
- Use instrumental audio or audio with clear melody
- Trim silence from the beginning/end
- Use higher audio quality (44.1kHz+ sample rate)
- Complex polyphonic music may have mixed results

### Limitations
- Works best for **monophonic** (single-note) melodies
- Polyphonic (multi-note) music will be simplified to a single melody line
- Very noisy audio may produce inaccurate results

### Next Steps
- Open the MIDI in a DAW (Ableton, FL Studio, Reaper, etc.)
- Edit and refine notes manually
- Add harmony, chords, and arrangement
- Export to different formats (MusicXML, etc.)

---

**Powered by [Spotify's Basic Pitch](https://github.com/spotify/basic-pitch)** — State-of-the-art automatic music transcription.